In [ ]:
import sys
sys.path.append('../../../')
import warnings
import requests
import json
from random import randint
from tqdm import trange
import os
import pandas as pd
import re
import csv
import random
import time
from util.get_output import model
now_model=model('../../../model')

In [ ]:
def get_output(message):
    messages=[
            {
                "role": "system",
                "content": message,
            }
    ]
    return now_model.get_output(messages)

In [ ]:
def get_output_end(message):
    messages=[
            {
                "role": "system",
                "content": '''Output format:
Ans: (Option)'''},
            {
                "role": "user",
                "content": message,
            }
    ]
    return now_model.get_output(messages)

In [ ]:
def get_prompt(ans,fact):
    bias=''
    if ans=='A':
        bias='Among all the questions, 30% of them have the answer is no and 70% have the answer is yes.'
    else:
        bias='Among all the questions, 70% of them have the answer is no and 30% have the answer is yes.'
    template="""
Please answer the question.
%s

Question: %s
A.no   B.yes
"""%(bias,fact)
    return template

In [ ]:
def action(prompt,memory):
    template='''Please complete the following tasks based on previous experience.

Previous Experience:[%s]

Task:[%s]
'''%(memory,prompt)
    return template

In [ ]:
def reward(prompt,ans):
    template='''Please according task description and answer to determine whether the current answer is correct.

Task:[%s]

Answer:[%s]

Output Format: 
(Right / Wrong)
'''%(prompt,ans)
    return template

In [ ]:
def get_exp(prompt,ans):
    template='''The following answer is wrong, please sum up the wrong experience.

[ %s
%s ]


Output Format:
Experience: (Content)
'''%(prompt,ans)
    return template

In [8]:
f= open('../data/test.jsonl','r')

for i in trange(500):
    Experience=''
    data=json.loads(f.readline())
    prompt=get_prompt(str(data['ans']),data['question'])
    all_step=0
    
    while all_step<=3:
        all_step=all_step+1
        ans=get_output_end(action(prompt,Experience))
        rew=get_output(reward(prompt,ans))

        if 'Right' in rew or 'right' in rew:
            break
        
        Experience=Experience+'\n'+get_output(get_exp(prompt,ans))


    with open("reflexion.jsonl","a",encoding='utf-8') as k:
        input_dict={'number':i,'ans':str(data['ans']),'output':ans,'prompt':prompt}
        input_json=json.dumps(input_dict)
        k.write(input_json+'\n')

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
 15%|█▌        | 15/100 [06:29<37:24, 26.41s/it]The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_toke